# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-lead

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-OCR
Updated
3 days ago
•
96.3k
•
691
moonshotai/Kimi-K2.5
Updated
1 day ago
•
203k
•
1.75k
openbmb/MiniCPM-o-4_5
Updated
about 3 hours ago
•
1k
•
536
Qwen/Qwen3-Coder-Next
Updated
3 days ago
•
18.7k
•
474
stepfun-ai/Step-3.5-Flash
Updated
about 20 hours ago
•
8.69k
•
453
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.26k
Qwen3-TTS Demo
🎙
1.26k
Transform text into natural-sounding speech with custom voices
Running
455
Demo Playground
⚡
455
Free platform to access multiple AI models
Running
on
A100
141
ACE-Step v1.5
🎵


In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-OCR\nUpdated\n3 days ago\n•\n96.3k\n•\n692\nmoonshotai/Kimi-K2.5\nUpdated\n1 day ago\n•\n203k\n•\n1.75k\nopenbmb/MiniCPM-o-4_5\nUpdated\nabout 3 hours ago\n•\n1k\n•\n536\nQwen/Qwen3-Coder-Next\nUpdated\n3 days ago\n•\n18.7k\n•\n474\nstepfun-ai/Step-3.5-Flash\nUpdated\nabout 20 hours ago\n•\n8.69k\n•\n453\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.26k\nQwen3-TTS D

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## Who We Are  
**Hugging Face** is a pioneering AI community and collaboration platform dedicated to building the future of machine learning. We provide a central hub where ML engineers, scientists, and AI enthusiasts around the world come together to share, discover, and innovate on open-source machine learning models, datasets, and applications. With millions of models and hundreds of thousands of datasets available, Hugging Face is at the forefront of the AI revolution, empowering a global community with cutting-edge tools to build open and ethical AI.

---

## Our Platform & Offerings  

### Collaboration Hub for Machine Learning  
- **Models:** Explore over 2 million open-source ML models across diverse modalities, including text, image, video, audio, and 3D.  
- **Datasets:** Access and share more than 500,000 datasets curated for various ML tasks.  
- **Spaces:** Deploy and interact with AI applications instantly in a user-friendly environment. Over 1 million applications are available.  
- **Community:** Connect with a vibrant, fast-growing network of developers, researchers, and users sharing insights and innovations.  

### Enterprise & Team Solutions  
Hugging Face offers tailored solutions for teams and enterprises to build AI responsibly at scale, including:  
- Enterprise-grade security with single sign-on (SSO), detailed audit logs, and granular access control.  
- Dedicated support with flexible contract options.  
- Advanced compute options such as ZeroGPU for scalable performance.  
- Private datasets viewer and private storage for enhanced collaboration and data safety.  
- Comprehensive analytics and billing management to track usage and optimize resources.

---

## Our Culture  

At Hugging Face, we are driven by a commitment to **openness, collaboration, and ethical AI development**. Our culture revolves around:  
- **Community-driven innovation:** We believe AI advances fastest and most responsibly when knowledge and resources are shared openly.  
- **Inclusivity:** Empowering the next generation of machine learning engineers and scientists globally.  
- **Transparency and Ethics:** Building AI technologies with responsibility at their core, ensuring fair opportunities and ethical standards.  
- **Continuous Learning:** We encourage growth and exploration at the edge of technology, supporting contributors and users alike in building their ML portfolios and profiles.

---

## Customers  

Our customers range from independent researchers and startups to large enterprises in industries spanning technology, healthcare, finance, and academia. They rely on Hugging Face to accelerate AI development, foster collaboration, and deploy cutting-edge solutions with confidence and security.

---

## Careers at Hugging Face  

Join us on the frontier of AI! Hugging Face seeks talented, passionate individuals eager to shape the future of machine learning. We offer opportunities across research, engineering, product, and community roles, with a strong emphasis on:  
- Impactful work contributing to open-source and ethical AI projects.  
- Collaborative and inclusive work environment.  
- Growth through challenging and innovative projects.  

Explore current openings on our [Careers page](https://huggingface.co/careers).

---

## Connect With Us  

- Website: [huggingface.co](https://huggingface.co)  
- GitHub: [github.com/huggingface](https://github.com/huggingface)  
- Twitter: [@huggingface](https://twitter.com/huggingface)  
- LinkedIn: [Hugging Face](https://linkedin.com/company/huggingface)  
- Discord Community: Join discussions, ask for help, or share your projects with fellow AI enthusiasts.

---

**Hugging Face** — Building the future of AI together.  
Join our community and start creating today!  
Sign up at [huggingface.co](https://huggingface.co)

---

*Colors associated with Hugging Face branding:*  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI community and platform dedicated to building the future of machine learning. It functions as a collaborative space where developers, researchers, and organizations come together to create, share, and improve machine learning models, datasets, and applications. Hugging Face empowers the community to accelerate innovation across all AI modalities including text, image, video, audio, and even 3D.

---

## What We Offer

- **Extensive Model Hub**: Browse over 2 million pre-trained machine learning models spanning diverse domains and tasks. Models are updated frequently to keep up with the latest advances.
- **Vast Dataset Repository**: Access 500,000+ datasets to power your AI projects, with continuous contributions from the community.
- **Spaces**: Deploy and explore 1 million+ interactive ML applications with free hosting, supporting real-time demos and AI services.
- **Community Collaboration**: Host and collaborate on unlimited public models, datasets, and AI applications with other researchers and developers worldwide.
- **Open Source Tools**: Leverage the HF open-source stack to accelerate your development workflow.
- **Enterprise Solutions**: Advanced paid compute and secure platforms tailored for teams and organizations, featuring enterprise-grade security and access control.

---

## Highlights

- **Trending Models**: Includes popular ML models like GLM-OCR for optical character recognition, Kimi-K2.5 for language understanding, and Qwen3-TTS for natural-sounding speech synthesis.
- **Multimodal Innovation**: Explore models and datasets across text, image, audio, video, and even 3D tasks.
- **AI Applications**: Use powerful tools like text-to-image generation, music composition models, and demo playgrounds—all run on high-performance infrastructure.
- **Community Activity**: Thousands of daily updates with new datasets, model improvements, and applications contributed by the AI community globally.

---

## Our Culture

Hugging Face thrives on openness, collaboration, and innovation. The company cultivates a vibrant community where contributors are encouraged to share their work, learn from others, and push the boundaries of AI technology. This commitment to open source fosters transparency and inclusivity, enabling rapid progress in machine learning research and applications.

---

## Our Customers

- **AI Researchers & Developers**: Build and share state-of-the-art models and datasets.
- **Enterprises & Teams**: Deploy scalable, secure AI solutions tailored for business needs.
- **Educational Institutions**: Use rich datasets and models for teaching and experimentation.
- **Startups & Innovators**: Access cutting-edge tools for rapid prototyping and development.

---

## Career Opportunities

Join a fast-growing, mission-driven company shaping the future of AI. Hugging Face offers exciting roles for:

- Machine Learning Engineers
- Research Scientists
- Software Developers
- Community Managers
- Enterprise Solutions Architects

If you're passionate about building collaborative AI technologies and contributing to an open ecosystem, Hugging Face provides an inspiring and supportive environment for your career growth.

---

## Connect with Hugging Face

- Website: https://huggingface.co  
- Join the platform and start collaborating today: [Sign Up](https://huggingface.co/join)  

---

*Hugging Face – Building the AI community that drives tomorrow’s innovations.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>